# Txt to pandas. Vectorize and rasterize
- We are just going to open a txt file and start doing shit
- we have the values on the txt and the coords in the csv
- both of them have a related field so we are going to mix then together
- After this. Get the value (temp / prec) and the coords and create a shp file
- Rasterize and clip the territory according to Emilia-Romagna
- Instructions on how to deal with the temp /prec valuees:
  - Precipitation: sum of all the values of the month / count of days per month which value has > 0.1
  - Tmax: monthly media of all the values
  - Tmin: monthly media of all the values
  - The last 30 years
- Data Sources 
  - https://dati.arpae.it/dataset/erg5-eraclito
  - https://dati.arpae.it/dataset/erg5-interpolazione-su-griglia-di-dati-meteo
  - https://drive.google.com/drive/folders/181nqJQDoFndicJsg4nCnnx6zvryjMOoj

In [1]:
import pandas as pd
import geopandas as gpd
import os
from shapely.geometry import Point

In [2]:
"""Create the df"""
# os.chdir(r"C:\Users\admin\Documents\03_data\Anna")
os.chdir(r"Z:\z_resources\ruben\anna")
"""Precipitation"""
data = pd.read_csv('Prec_ERG5_Eraclito.txt', header=None)

"""Temperature max"""
# data = pd.read_csv('tmax_erg5_eraclito.txt', header=None)

"""Temperature min"""
# data = pd.read_csv('tmin_erg5_eraclito.txt', header=None)

"""Template with the codes that refer to the coords"""
template = pd.read_csv('Erg5_Eraclito_structure - Foglio1_1.csv')



In [3]:
"""Split the data into several columns"""
data = data[0].str.split(expand=True) #separate the code with the value into different columns
data.rename(columns={1: 'Value'}, inplace=True) #give a name "Value" to the 1 column
data["Code"] = data[0].str[0:5].str.lstrip("0") #extract the total code and remove 0 from the left on the code
data["Date"] = data[0].str[5:] #take the date from the code
data = data.drop([0], axis = 1) #delete the chunk we don't need
data["Code"] = data["Code"].astype(int) #WARNING: It is important that the matching columns are INT!!!!

cols = ["Code","Date","Value"] #reorder the columns
data = data[cols]
data["Value"] = data["Value"].astype(float) #transform the values into float (just in case)


In [4]:
#check internal data type, both Code have to be int
print(data.dtypes, "\n", data.size)
print(template.dtypes, "\n", template.size)

Code       int32
Date      object
Value    float64
dtype: object 
 74357595
Code                 int64
Lat (WGS84)         object
Lon (WGS84)         object
X (WGS84 UTM32N)     int64
Y (WGS84 UTM32N)     int64
Height              object
dtype: object 
 6786


In [5]:
"""Check the code values"""
unique_data_codes = list(data.Code.unique())
unique_template_codes = list(template.Code)
#in total are 1131 elements which are the same at the template

In [6]:
data[0:2]

,Code,Date,Value
0,1960,19610101,3.9
1,1960,19610102,33.7


In [7]:
template[0:2]

,Code,Lat (WGS84),Lon (WGS84),X (WGS84 UTM32N),Y (WGS84 UTM32N),Height
0,19,"4,458,750","9,133,525",510599,4937137,"1120,7"
1,20,"4,454,250","9,133,525",510607,4932139,"985,3"


In [5]:
"""now, do the merge of both elements"""
#Warning: both mixing columns have to be the same data type (int)
result = pd.merge(data, template, on = "Code", how='inner') #inner takes the ones that match, outer everyting
# result = pd.concat([data, template], axis=0)

In [6]:
# result.size
# inner 198286920
# outer 198286920
# result.dtypes
result.head()

,Code,Date,Value,Lat (WGS84),Lon (WGS84),X (WGS84 UTM32N),Y (WGS84 UTM32N),Height
0,1960,19610101,3.9,"4,364,250","12,159,925",754867,4837021,"955,1"
1,1960,19610102,33.7,"4,364,250","12,159,925",754867,4837021,"955,1"
2,1960,19610103,0.0,"4,364,250","12,159,925",754867,4837021,"955,1"
3,1960,19610104,58.2,"4,364,250","12,159,925",754867,4837021,"955,1"
4,1960,19610105,7.6,"4,364,250","12,159,925",754867,4837021,"955,1"


In [7]:
"""Format the table, convert everything to object, and the value to float so we can groupby value"""
result["Value"] = result["Value"].astype(float).round(decimals = 1)
result["Code"] = result["Code"].astype(object)
#we are using UTM32N
result["X (WGS84 UTM32N)"] = result["X (WGS84 UTM32N)"].astype(object)
result["Y (WGS84 UTM32N)"] = result["Y (WGS84 UTM32N)"].astype(object)
#we were trying to use the wgs84 coordinates, but the values are wrong
# result["Lat (WGS84)"] = result["Lat (WGS84)"].apply(lambda x: str(x).split()[0].replace(',','')) #remove the commas to transform later the str to float
# result["Lon (WGS84)"] = result["Lon (WGS84)"].apply(lambda x: str(x).split()[0].replace(',',''))

result = result.rename(columns={"Lat (WGS84)":"Lat_WGS84", "Lon (WGS84)":"Lon_WGS84", "X (WGS84 UTM32N)":"X_WGS84_UTM32N", "Y (WGS84 UTM32N)":"Y_WGS84_UTM32N"})
result.dtypes

Code               object
Date               object
Value             float64
Lat_WGS84          object
Lon_WGS84          object
X_WGS84_UTM32N     object
Y_WGS84_UTM32N     object
Height             object
dtype: object

In [11]:
"""this part is for checking the result and see if we do the sum correctly"""
filtered_data = result[result['Date'].str.startswith("202001")]
filtered_data_1 = filtered_data.groupby(['Code','Lat_WGS84','Lon_WGS84','X_WGS84_UTM32N','Y_WGS84_UTM32N','Height'], as_index=False).sum().round(1)
filtered_data_2 = filtered_data.groupby(['Lat_WGS84','Lon_WGS84','X_WGS84_UTM32N','Y_WGS84_UTM32N','Height'], as_index=False).sum().round(1)
filtered_data_1.head()


,Code,Lat_WGS84,Lon_WGS84,X_WGS84_UTM32N,Y_WGS84_UTM32N,Height,Value
0,19,"4,458,750","9,133,525",510599,4937137,"1120,7",136.9
1,20,"4,454,250","9,133,525",510607,4932139,"985,3",148.8
2,21,"4,449,750","9,133,525",510615,4927140,"773,0",170.4
3,52,"4,490,250","9,196,575",515519,4972139,"456,6",48.1
4,53,"4,485,750","9,196,575",515531,4967140,"649,2",48.3


In [8]:
"""work with the dates / we have to use all the dates"""
dates = list(result.Date.unique()) #here is all the list of all the dates

#here is the list of the month dates
month_date = []
for date in dates:
    if (date[0:6]) in month_date: #we take the year and the month
        pass
    else:
        month_date.append(date[0:6])
        
month_date[0:5]
#date[0:4] + "_" + date[4:6]

['196101', '196102', '196103', '196104', '196105']

In [11]:
"""Precipitation data"""
"""Filter the main df with the corresponding date"""
os.chdir(r"Z:\z_resources\ruben\anna\prec_sum_erg5_eraclito")

for i in month_date[696:720]: #720 total elements. 12 months x 30 years = 360
    result_date = result[result["Date"].str.startswith(i)] #filter the main table with the date
    #groupby the value
    result_date = result_date.groupby(['Code','Lat_WGS84','Lon_WGS84','X_WGS84_UTM32N','Y_WGS84_UTM32N','Height'], as_index=False).sum().round(1)
    #create the geometry 
    result_date['geometry'] = result_date.apply(lambda x: Point((float(x.X_WGS84_UTM32N), float(x.Y_WGS84_UTM32N))), axis=1) #we create the geometry column
    result_date = gpd.GeoDataFrame(result_date, geometry='geometry')
    #assign a projection and reproject to 4326
    result_date.crs = "EPSG:32632"
    result_date = result_date.to_crs("EPSG:4326")
    #export the result
    result_date.to_file('Arpae_precipitation_sum_of_emilia_romagna_{}.shp'.format(i[0:4] + "_" + i[4:6]),index=False) #here we do the shp export
    # result_date.to_csv('Prec_ERG5_Eraclito_sum_{}.csv'.format(i),index=False) #here we do the csv export
   

c:\Users\admin\.conda\envs\gdal_env\lib\site-packages\ipykernel_launcher.py:16: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  app.launch_new_instance()


In [70]:
"""Precipitation data / count of days per month which value has > 0.1"""
"""Filter the main df with the corresponding date"""
os.chdir(r"C:\Users\admin\Documents\03_data\Anna\Prec_count_ERG5_Eraclito")

for i in month_date[360:720]: #12 months x 30 years = 360
    result_date = result[result["Date"].str.startswith(i)] #filter the main table with the date
    #filter the data with less than 0.1 value
    result_date = result_date[result_date["Value"] < 0.1]
    #groupby the value and count it
    result_date = result_date.groupby(['Lat_WGS84','Lon_WGS84','X_WGS84_UTM32N','Y_WGS84_UTM32N','Height'], as_index=False)['Value'].count()
    # create the geometry 
    result_date['geometry'] = result_date.apply(lambda x: Point((float(x.X_WGS84_UTM32N), float(x.Y_WGS84_UTM32N))), axis=1) #we create the geometry column
    result_date = gpd.GeoDataFrame(result_date, geometry='geometry')
    #assign a projection and reproject to 4326
    result_date.crs = "EPSG:32632"
    result_date = result_date.to_crs("EPSG:4326")
    #export the result
    result_date.to_file('Arpae_precipitation_day_count_of_emilia_romagna_{}.shp'.format(i[0:4] + "_" + i[4:6]),index=False) #here we do the shp export
    # result_date.to_csv('Prec_ERG5_Eraclito_sum_{}.csv'.format(i),index=False) #here we do the csv export

c:\Users\admin\.conda\envs\gdal_env\lib\site-packages\ipykernel_launcher.py:18: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.


In [64]:
filtered_result = result_date[result_date["Value"] < 0.1]
test = filtered_result.groupby(['Lat_WGS84','Lon_WGS84','X_WGS84_UTM32N','Y_WGS84_UTM32N','Height'], as_index=False)['Value'].count()
# test = result_date.groupby(['Lat_WGS84','Lon_WGS84','X_WGS84_UTM32N','Y_WGS84_UTM32N','Height'], as_index=False).filter(lambda x: x['Value'].mean() > 0.1)
# test.filter(lambda x: x['Value'].mean() > 0.1)
# test.head()
# test = result_date[result_date["Value"] < 0.1]

In [14]:
"""Temperature max data"""
os.chdir(r"C:\Users\admin\Documents\03_data\Anna\tmax_erg5_eraclito")

for i in month_date[360:720]: #12 months x 30 years = 360
    result_date = result[result["Date"].str.startswith(i)] #filter the main table with the date
    #groupby the value
    result_date = result_date.groupby(['Lat_WGS84','Lon_WGS84','X_WGS84_UTM32N','Y_WGS84_UTM32N','Height'], as_index=False).mean().round(1)
    #create the geometry 
    result_date['geometry'] = result_date.apply(lambda x: Point((float(x.X_WGS84_UTM32N), float(x.Y_WGS84_UTM32N))), axis=1) #we create the geometry column
    result_date = gpd.GeoDataFrame(result_date, geometry='geometry')
    #assign a projection and reproject to 4326
    result_date.crs = "EPSG:32632"
    result_date = result_date.to_crs("EPSG:4326")
    #export the result
    result_date.to_file('Arpae_mean_temp_max_of_emilia_romagna_{}.shp'.format(i[0:4] + "_" + i[4:6]),index=False) #here we do the shp export
    # result_date.to_csv('Prec_ERG5_Eraclito_sum_{}.csv'.format(i),index=False) #here we do the csv export
   

c:\Users\admin\.conda\envs\gdal_env\lib\site-packages\ipykernel_launcher.py:15: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  from ipykernel import kernelapp as app


In [13]:
"""Temperature min data"""
os.chdir(r"C:\Users\admin\Documents\03_data\Anna\tmin_erg5_eraclito")

for i in month_date[360:720]: #12 months x 30 years = 360
    result_date = result[result["Date"].str.startswith(i)] #filter the main table with the date
    #groupby the value
    result_date = result_date.groupby(['Lat_WGS84','Lon_WGS84','X_WGS84_UTM32N','Y_WGS84_UTM32N','Height'], as_index=False).mean().round(1)
    #create the geometry 
    result_date['geometry'] = result_date.apply(lambda x: Point((float(x.X_WGS84_UTM32N), float(x.Y_WGS84_UTM32N))), axis=1) #we create the geometry column
    result_date = gpd.GeoDataFrame(result_date, geometry='geometry')
    #assign a projection and reproject to 4326
    result_date.crs = "EPSG:32632"
    result_date = result_date.to_crs("EPSG:4326")
    #export the result
    result_date.to_file('Arpae_mean_temp_min_of_emilia_romagna_{}.shp'.format(i[0:4] + "_" + i[4:6]),index=False) #here we do the shp export
    # result_date.to_csv('Prec_ERG5_Eraclito_sum_{}.csv'.format(i),index=False) #here we do the csv export

c:\Users\admin\.conda\envs\gdal_env\lib\site-packages\ipykernel_launcher.py:15: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  from ipykernel import kernelapp as app


# Interpolation of the data

In [12]:

from osgeo import gdal

In [13]:
"""Get a list of the shp files"""
#Get a list of the files inside the folder
path = r"Z:\z_resources\ruben\anna\prec_sum_erg5_eraclito"
shp_file_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
for file in os.listdir(path):
    if ".shp" in file:
        if file not in shp_file_list:
            shp_file_list.append(file)
    else:
        pass
print(shp_file_list[:12])

['arpae_precipitation_sum_of_emilia_romagna_2019_01.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_02.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_03.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_04.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_05.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_06.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_07.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_08.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_09.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_10.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_11.shp', 'arpae_precipitation_sum_of_emilia_romagna_2019_12.shp']


In [15]:
"""use the shp files to do the interpolation"""
#edit the folder according to your preference
os.chdir(r"Z:\z_resources\ruben\anna\prec_sum_erg5_eraclito")
for file in shp_file_list:
    # dataSource = ogr.Open(file, 0) #0 = read mode
    # layer = dataSource.GetLayer()
    output = file.replace(".shp",".tif")
    """elect the interpolation method"""
    # nn = gdal.Grid(output, file, zfield="Value", algorithm = "invdist:power=3:radius1=0.07:radius2=0.07:nodata=0", #invert distance interpolation
    nn = gdal.Grid(output, file, zfield="Value", algorithm = "nearest", #nearest neighbor interpolation
    
    width = 100,
    height = 100
    # :power=3:radius1=2000:radius2=2000"
    )
    nn = None
    

In [18]:
"""Get a list of the tif files"""
#Get a list of the files inside the folder
#edit the folder according to your preference
path = r"Z:\z_resources\ruben\anna\prec_sum_erg5_eraclito\input_raster"
tif_file_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
for file in os.listdir(path):
    if ".tif" in file:
        if file not in tif_file_list:
            tif_file_list.append(file)
    else:
        pass
print(tif_file_list[:12])

['arpae_precipitation_sum_of_emilia_romagna_2019_01.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_02.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_03.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_04.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_05.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_06.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_07.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_08.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_09.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_10.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_11.tif', 'arpae_precipitation_sum_of_emilia_romagna_2019_12.tif']


In [19]:
"""Do the clipping area"""
#edit the folders according to your preference
os.chdir(r"Z:\z_resources\ruben\anna\prec_sum_erg5_eraclito")
input_path = r"Z:\z_resources\ruben\anna\prec_sum_erg5_eraclito\input_raster"
cutfile = r"Z:\z_resources\ruben\anna\italy_region.shp"
for file in tif_file_list:
    # print(path + "\\" + file)
    ds = gdal.Open(input_path + "\\" + file)
    ds_clipped = gdal.Warp(file, 
        ds,
        cutlineDSName=cutfile,
        cropToCutline=True,
        dstNodata = 0,
        creationOptions = ['COMPRESS=DEFLATE','TILED=YES','BLOCKXSIZE=128','BLOCKYSIZE=128']) 

        #input_raster_filepath, output_raster_filepath, cutlineDSName=road_buffered_filepath, cropToCutline=True)
    ds_clipped = None